## "Optimizing Emergency Hospital (EH) Facility Location Using the P-median model"

## Objective and Prerequisites

The primary goal of this research is to optimize the placement of Emergency Hospital (EH) facilities in Stockholm. The study develops a two-stage computational artifact: first, a Poisson Generalized Linear Model (GLM) is used to estimate demographic-specific risk ($\lambda$); second, a Capacitated P-median model is applied to minimize the total system risk.

## Motivation
The placement of Emergency Health Service (EHS) facilities is a high-stakes decision that directly impacts public safety and urban resilience. Traditional facility location models often rely on a "one-size-fits-all" approach, focusing purely on minimizing the average travel distance for the entire population. However, in a complex urban environment like Stockholm, geographical proximity does not always equate to effective healthcare accessibility.

This research is motivated by three critical gaps in standard urban planning:

Demographic Vulnerability: Different age groups have vastly different healthcare needs. An emergency for an infant or an elderly person carries a higher clinical risk during transit than a similar emergency for a healthy adult. A model that treats all residents as identical data points fails to protect the most vulnerable.

The "15-Minute City" Evolution: As Stockholm moves toward a "15-minute city" model, the goal is to ensure essential services are within a short reach. However, "reach" must be defined by Risk, not just meters. We need a system that prioritizes locations where the travel-time-to-risk ratio is highest.

Resource Constraints: Hospitals have finite capacities. Simply identifying the "best" spot is insufficient if that facility becomes overwhelmed. There is a need for a mathematical framework that balances optimal location with operational reality (capacity constraints).

By integrating Poisson-based Statistical Inference with Mixed-Integer Programming (MIP), this thesis moves beyond simple logistics. We transform a standard warehouse-style location problem into a Life-Critical Decision Support System. The motivation is to provide urban planners with a tool that doesn't just find the nearest hospital, but the safest configuration for the entire city’s demographic profile.

## Problem Description

The Stockholm healthcare administration needs to determine the optimal locations for a set of Emergency Health Service (EHS) facilities to serve a diverse population across various zip codes. The challenge is not merely a logistics problem of minimizing distance; it is a public health challenge of minimizing clinical risk during transit, particularly for vulnerable age groups.
In this scenario, we consider a set of potential hospital locations (candidates) and a set of demand points (zip codes). Each zip code contains a population divided into three demographic cohorts: Infants (0-5 years), Elderly (65+ years), and the General Population.
The Core Challenges

•	Heterogeneous Demand: Different age groups have different "visit intensities." For example, an elderly person may statistically require emergency services more frequently than a healthy young adult.

•	Distance-Dependent Risk: The danger of a medical emergency increases as the travel distance to a hospital increases. However, this "risk" is higher for a high-needs patient than for a low-needs patient over the same distance.

•	Facility Constraints: Each hospital candidate has a maximum Capacity (the total number of patients it can effectively serve). We cannot simply assign everyone to the single "best" hospital; we must distribute the population based on available resources.

•	Strategic Budgeting ($p$): Urban planners have a limited budget and can only open a fixed number of facilities, denoted as $p$. We need to know which $p$ locations provide the maximum reduction in total city-wide risk.

## Solution Approach

Mathematical programming is a declarative approach where the modeler formulates a mathematical optimization model that captures the key aspects of a complex public health challenge. The Gurobi Optimizer solves such models using state-of-the-art mathematics and computer science, ensuring that we find the absolute best (globally optimal) hospital locations under given constraints.A mathematical optimization model has five components, namely:

1. Data Preparation: Creating the Stockholm spatial matrix.

2. Risk Estimation: Using the Poisson GLM to predict demand by Age/Gender.

3. Optimization: Setting up the Gurobi model with Decision Variables ($x, y$) and Constraints.

4. Sensitivity Analysis: Visualizing the trade-offs between $p$, Radius, and Risk.

In this study, we utilize a two-stage approach. First, we use a Poisson Mixed Model to infer the parameters for clinical risk. Second, we present a Mixed-Integer Programming (MIP) formulation for the facility location problem to decide the optimal spatial configuration.

##  Model Formulation

## Sets and Indices

$i \in I$: Index and set of demand points (combination of Zip Code and Age Group).

$j \in J$: Index and set of candidate hospital (facility) locations.

## Parameters

$p \in \mathbb{Z}^+$: The fixed number of Emergency Health Service (EHS) facilities to be opened in the network.

$R_{i,j} \in \mathbb{R}^+$: The calculated Clinical Risk for demand point $i$ if assigned to hospital $j$. This is derived from the Poisson Generalized Linear Model (GLM) as:
$$R_{i,j} = \frac{\lambda_{i,j}}{\max(d_{i,j}, 0.5)}

$$where $\lambda_{i,j}$ is the expected demand (predicted visits) and $d_{i,j}$ is the Haversine distance in kilometers.

$Pop_{i} \in \mathbb{R}^+$: The population count of the specific demographic cohort (Age/Gender) at demand point $i \in I$.

$Cap_{j} \in \mathbb{R}^+$: The maximum annual patient capacity of candidate hospital $j \in J$.

$d_{i,j} \in \mathbb{R}^+$: The geographical distance between demand point $i$ and candidate hospital $j$, calculated using the Haversine formula. (Note: You use this in your radius constraint, so it should be defined as a parameter.)

## Decision Variables

$y_{j} \in \{0, 1 \}$: This binary variable is equal to 1 if we open a facility at candidate location $j \in J$; and 0 otherwise.

$x_{i,j} \in \{0, 1 \}$: This binary variable is equal to 1 if demand point $i \in I$ is assigned to facility $j \in J$; and 0 otherwise.

## Objective Function

Total Risk Minimization. The goal is to minimize the total system-wide risk. This is the sum of the risk-weighted assignments across all zip codes and age groups. By minimizing this objective, the model identifies locations that reduce the clinical danger for the most vulnerable populations.

$$\text{Min} \quad Z = \sum_{i \in I} \sum_{j \in J} R_{i,j} \cdot x_{i,j}$$


## Constraints

Single Assignment. Each demand point (zip code + age group) must be assigned to exactly one EHS facility to ensure full coverage of the population.

$$\sum_{j \in J} x_{i,j} = 1, \quad \forall i \in I$$
 ## Facility Linkage. 
 
 A demand point can only be assigned to a candidate hospital if that hospital has been selected (opened).

 $$x_{i,j} \le y_{j}, \quad \forall i \in I, \forall j \in J$$


 ## Facility Count. 
 
 Exactly $p$ facilities must be opened across the Stockholm region, representing the strategic budget or resource limit.

 $$\sum_{j \in J} y_{j} = p$$
 
 ## Capacity Limit. 
 
 The total population assigned to a hospital $j$ must not exceed its operational capacity.

 $$\sum_{i \in I} Pop_{i} \cdot x_{i,j} \leq Cap_{j} \cdot y_{j}, \quad \forall j \in J$$

## Python Implementation

This study considers a set of potential Emergency Health Service (EHS) candidates and demand points (Zip Codes) across Stockholm. The coordinates and specific attributes for the healthcare facilities are provided in the following table.

Hospital Name |Coordinates (Lat, Lon) | Capacity (Patients)
| --- | --- |  --- |
|Karolinska      | (59.3529, 18.0348)       | 15,000 |
|Nacka           | (59.3117, 18.1189)       | 10,000 |
|Södersjukhuset  | (59.3047, 18.0511)       | 20,000 |
|Kista           | (59.4035, 17.9424)       | 12,000 |
|Central Station | (59.3326, 18.0649)       |  8,000 |
|Bromma Area     | (59.3227, 17.9752)       |  9,000 |
|Odenplan Clinic | (59.3421, 18.0519)       |  7,000 |
|Enskede Health  | (59.2842, 18.0831)       |  8,500 |


## Data Pre-processing

We utilize the Haversine formula to calculate the Great-Circle distance between each Zip Code centroid and every candidate hospital site. To estimate the medical demand, we categorize the population into three cohorts: Infants (0-5), Elderly (65+ ), and General Population, each with distinct emergency visit rates.

In [ ]:
#%pip install gurobipy
#%pip install gmplot

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
from math import radians, sin, cos, sqrt, atan2
import gmplot

## 1. Data Preparation (Enhanced Granularity)

In the data preparation phase, a spatial interaction matrix was constructed by cross-joining 10 Stockholm demand points with 8 candidate EHS facilities. Population data was disaggregated by age and gender to allow for high-granularity risk estimation, and travel distances were computed using the Haversine formula to ensure geodetic accuracy.

In [ ]:
# ==========================================
# 1. ZIP CODE DATA (UNCHANGED)
# ==========================================
zip_data = {
    'zip_code':['11120','11210','11330','11420','11550','11610','11720','11830','12010','12130'],
    'lat':[59.332, 59.331, 59.344, 59.340, 59.348, 59.311, 59.315, 59.318, 59.298, 59.291],
    'lon':[18.064, 18.021, 18.049, 18.078, 18.102, 18.074, 18.032, 18.051, 18.081, 18.121],
    'pop_0_5_M':[100,75,150,60,125,200,90,110,155,140],
    'pop_0_5_F':[100,75,150,60,125,200,90,110,155,140],
    'pop_65_plus_M':[240,210,280,180,260,380,250,290,340,310],
    'pop_65_plus_F':[260,240,320,220,290,420,270,320,360,340],
    'pop_gen_M':[1000,900,1250,750,1100,1500,950,1050,1200,1150],
    'pop_gen_F':[1000,900,1250,750,1100,1500,950,1050,1200,1150]
}
df_zip = pd.DataFrame(zip_data)
df_zip.to_csv("zip_data.csv", index=False)
df_zip

In [ ]:
# ==========================================
# 2. HOSPITAL DATA (FIXED COLUMN NAMES)
# ==========================================
ehs_candidates = [
    (59.3529, 18.0348, "Karolinska Solna", 160000), 
    (59.3117, 18.1189, "Nacka Sjukhus", 45000),
    (59.3047, 18.0511, "Södersjukhuset", 145000),
    (59.4035, 17.9424, "Kista Clinic", 35000),
    (59.3326, 18.0649, "City Emergency", 60000),
    (59.2331, 18.0224, "Danderyds sjukhus", 90000),
    (59.3421, 18.0519, "Odenplan Clinic", 25000),
    (59.2842, 18.0831, "Enskede Health", 30000)
]

df_hosp = pd.DataFrame(ehs_candidates, columns=["lat", "lon", "hospital", "capacity"])
df_hosp.to_csv("ehs_candidates.csv", index=False)
df_hosp

In [ ]:
# ==========================================
# 3. HAVERSINE FUNCTION 
# ==========================================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

In [ ]:
# ==========================================
# 4. DISTANCE MATRIX 
# ==========================================
df_zip["key"], df_hosp["key"] = 1, 1
df = pd.merge(df_zip, df_hosp, on="key", suffixes=("_zip", "_hosp"))

df["distance"] = haversine(
    df["lat_zip"], df["lon_zip"],
    df["lat_hosp"], df["lon_hosp"]
)

## 5. Risk Estimation (Poisson with Gender)

To establish a robust objective function, a Poisson Generalized Linear Model (GLM) was employed to estimate emergency demand ($\lambda$). By incorporating age, gender, and distance as covariates with a population offset, the model accounts for heterogeneous healthcare needs across Stockholm. The resulting risk metric provides a distance-decayed demand signal that prioritizes facility placement in regions with high-vulnerability demographic clusters

In [ ]:
# ==========================================
# 5. Risk Estimation (Poisson with Gender)
# ==========================================

# Step 2.1: Data Melting (Reshaping to Long Format)
id_cols = ["zip_code", "hospital", "distance", "capacity"]
val_cols = [c for c in df_zip.columns if 'pop_' in c]

# This step transforms the data into 60 unique demand points per hospital
df_long = df.melt(id_vars=id_cols, value_vars=val_cols, 
                  var_name="demographic", value_name="population")

# Step 2.2: Demographic Feature Engineering
# Splitting 'pop_0_5_M' into 'pop_0_5' (Age Group) and 'M' (Gender)
df_long[['age_group', 'gender']] = df_long['demographic'].str.rsplit('_', n=1, expand=True)

# Mapping base rates for visit probability based on age group
rate_map = {"pop_0_5": 0.32, "pop_65_plus": 0.55, "pop_gen": 0.18}
df_long["rate"] = df_long["age_group"].map(rate_map)

# Calculating raw visits and setting up the Log-Population offset for Poisson GLM
df_long["visits"] = df_long["population"] * df_long["rate"]
df_long["log_pop"] = np.log(np.maximum(df_long["population"], 1))

# Step 2.3: Poisson GLM Model Fitting
# Formula: Visits explained by Distance, Age, Hospital, and Gender
formula = "visits ~ distance + C(age_group) + C(gender) + C(hospital)"
model = smf.glm(formula=formula, 
                data=df_long, 
                family=sm.families.Poisson(), 
                offset=df_long["log_pop"])

result = model.fit()

# Step 2.4: Prediction and Risk Scoring
# 'lambda' is the predicted demand rate from the model
df_long["lambda"] = result.predict(df_long)


## 6. Multi-Parameter Sensitivity Analysis ( Decision Variable)

The optimization phase employs a Mixed-Integer Programming (MIP) framework to evaluate resource allocation across the Stockholm metropolitan area. The model iteratively solves for different facility counts ($p$), allowing for a comparative analysis of how infrastructure investment (the number of open hospitals) impacts the aggregate clinical risk. By minimizing a risk-weighted distance objective function, the framework identifies the optimal spatial configuration of emergency health services while respecting hospital capacity constraints and population demand patterns.

In [ ]:
# ==========================================
# 6. Precompute Cost and Distance Matrices
# ==========================================

# Step 4.0: Define Group Mappings and Lists
group = {
    'pop_0_5_M': '0-5_male', 
    'pop_0_5_F': '0-5_female',
    'pop_65_plus_M': '65+_male', 
    'pop_65_plus_F': '65+_female',
    'pop_gen_M': 'general_male', 
    'pop_gen_F': 'general_female'
}

# Creating lists for easy iteration
group_list = list(group.values())
original_cols = list(group.keys())

# Step 4.1: Create Lookup Dictionaries for Coordinates
# Converting DataFrames to dictionaries for faster O(1) access
zip_coords = df_zip.set_index('zip_code')[['lat','lon']].to_dict('index')
hosp_coords = df_hosp.set_index('hospital')[['lat','lon']].to_dict('index')

# Step 4.2: Prepare Demand Points (60 points: 10 Zips * 6 Groups)
demand_points = df_long[['zip_code', 'demographic', 'population']].drop_duplicates().copy()
demand_points['group'] = demand_points['demographic'].map(group)

# Step 4.3: Create Risk and Population Lookups
# Risk is the mean lambda for each (zip, demographic) pair
risk_lookup = df_long.groupby(['zip_code', 'demographic'])['lambda'].mean().to_dict()

# Population lookup for the Capacity Constraint
pop_lookup = demand_points.set_index(['zip_code', 'group'])['population'].to_dict()

# Step 4.4: Initialize Sets and Matrices
zip_list = [str(z) for z in df_zip['zip_code'].unique()]
hospitals = df_hosp['hospital'].unique()
J = range(len(hospitals))

cost_ij = {}
dist_ij = {}

# Step 4.5: Calculate Distance and Risk-Weighted Cost for all (i, j) pairs
# Total iterations: 60 demand points * 8 hospitals = 480 calculations
for i in range(len(demand_points)):
    row_i = demand_points.iloc[i]
    zip_i = row_i['zip_code']
    demo_i = row_i['demographic'] 
    group_i = row_i['group'] 
    
    # Define the 60 unique Demand Point Tuple (I-tuple)
    i_tuple = (zip_i, group_i) 

    for j in range(len(hospitals)):
        hosp_j = hospitals[j]
        
        # Calculate Haversine Distance
        d_ij = haversine(zip_coords[zip_i]['lat'], zip_coords[zip_i]['lon'], 
                         hosp_coords[hosp_j]['lat'], hosp_coords[hosp_j]['lon'])
        
        # Get precomputed Risk (Lambda)
        risk_ij = risk_lookup.get((zip_i, demo_i), 0)
        
        # Store results in matrices
        dist_ij[i_tuple, j] = d_ij 
        cost_ij[i_tuple, j] = risk_ij * d_ij

print(f"Precomputation complete: {len(dist_ij)} pairs generated.")

In [ ]:
I_tuples = list(set(key[0] for key in cost_ij.keys()))
print("Size of I_tuples:", len(I_tuples))
print("Sample I_tuples:", I_tuples[:6])

In [ ]:
print(list(cost_ij.keys())[:5])

In [ ]:
# ==========================================
# 7. Multi-Parameter Optimization Loop 
# ==========================================

# --- Parameter Initialization ---
p_values = [3, 4, 5]
results_matrix = {}
runtimes = {}

# Use the unique hospital names for lookup and indexing
hospitals_list = df_hosp['hospital'].unique()

J = sorted(list(set([key[1] for key in cost_ij.keys()])))

for p_val in p_values:
    # Initialize Gurobi model
    m = gp.Model(f"EHS_P{p_val}")
    m.Params.OutputFlag = 0  # Suppress Gurobi console output

    # Step 7.1: Create Set I (Demand Points)----
    I_tuples = list({key[0] for key in cost_ij.keys()})

    # --- Step 7.2: Define Variables as a Set Product ---
    valid_pairs = set(cost_ij.keys())
    x = m.addVars(valid_pairs, vtype=GRB.BINARY, name="x")
    y = m.addVars(J, vtype=GRB.BINARY, name="y")
    

    # -----Step 7.3: Objective Function-----

    m.setObjective(
    gp.quicksum(cost_ij[key] * x[key] for key in valid_pairs),
    GRB.MINIMIZE
)

    # --- Step 7.4: Constraints ---
    
    # 7.4.1. Coverage: Each demand point i must be assigned to one hospital j
    for i in I_tuples:
        m.addConstr(
        gp.quicksum(x[key] for key in valid_pairs if key[0] == i) == 1
    )

    # 7.4.2. Linkage: Assignment only possible to open hospitals
    for key in valid_pairs:
        i, j = key
        m.addConstr(x[key] <= y[j])

    # 7.4.3. Budget/P-facility: Exactly P hospitals must be open
    m.addConstr(y.sum() == p_val, name="P_Facility_Constraint")

    # 7.4.4. Capacity: Total population assigned to hospital j cannot exceed its capacity
    for j in J:
        h_name = hospitals_list[j]
        capacity_j = df_hosp[df_hosp['hospital'] == h_name]['capacity'].values[0]
    
        m.addConstr(
        gp.quicksum(
            pop_lookup.get(key[0], 0) * x[key]
            for key in valid_pairs if key[1] == j
        ) <= capacity_j * y[j]
    )

    # --------7.5: Optimize Model----------
    m.optimize()

    # Store and Print Results
    if m.status == GRB.OPTIMAL:
        results_matrix[p_val] = m.objVal
        runtimes[p_val] = m.Runtime
        print(f"\n>>> RESULTS FOR P = {p_val} <<<")
        print(f"Objective Value (Z): {m.objVal:.4f}")
        print(f"Solving Time: {m.Runtime:.4f} seconds")
        
        # Show which hospitals were opened
        opened = [hospitals_list[j] for j in J if y[j].X > 0.5]
        print(f"Opened Hospitals: {opened}")
    else:
        print(f"P = {p_val}: Optimization did not find an optimal solution. Status: {m.status}")

## 8.Sensitivity Analysis:

The final stage of the artifact generates a high-resolution GIS visualization using gmplot. This map illustrates the spatial distribution of the 60 demographic groups across Stockholm and their optimal assignments to the selected hospital facilities. By plotting the connection between demand points and hospitals, the visualization confirms that the model successfully adheres to capacity constraints while minimizing travel distances for high-risk populations.

In [ ]:
# ==========================================
# 8. Sensitivity Analysis & Visualization
# ==========================================

if p_val == max(p_values):
    gmap = gmplot.GoogleMapPlotter(59.3293, 18.0686, 12)

    # (Offset)
    offsets = [
        (0.0007, 0.0007), (-0.0007, -0.0007), 
        (0.0007, -0.0007), (-0.0007, 0.0007),
        (0.001, 0), (0, 0.001)
    ]
    
    # 60 demand point (I_tuples) plot
    for idx, i_tuple in enumerate(I_tuples):
        z_code, g_name = i_tuple
        base_lat = zip_coords[z_code]['lat']
        base_lon = zip_coords[z_code]['lon']
        
        # Jittered Coordinates
        off_lat, off_lon = offsets[idx % len(offsets)]
        jittered_lat = base_lat + off_lat
        jittered_lon = base_lon + off_lon
        
        # 'scatter'
        # 'title' 
        gmap.marker(
            jittered_lat, 
            jittered_lon, 
            color='blue', 
            title=f"ZIP: {z_code} | Group: {g_name}"
        )

    # Hospital(Red)
    for index, row in df_hosp.iterrows():
        gmap.marker(
            row['lat'], 
            row['lon'], 
            color='red', 
            title=f"Hospital: {row['hospital']}"
        )

    # Assignment line (Blue lines)
    for key in x.keys():
        if x[key].X > 0.5:
            (z_code, g_name), j_idx = key
            i_idx = I_tuples.index((z_code, g_name))
            off_lat, off_lon = offsets[i_idx % len(offsets)]
            
            start_lat = zip_coords[z_code]['lat'] + off_lat
            start_lon = zip_coords[z_code]['lon'] + off_lon
            
            h_name = hospitals_list[j_idx]
            end_lat = hosp_coords[h_name]['lat']
            end_lon = hosp_coords[h_name]['lon']

            gmap.plot([start_lat, end_lat], [start_lon, end_lon], 
                      color='cornflowerblue', edge_width=1.0)

    # File
    gmap.draw(f"Stockholm_EH_Map_P{p_val}.html")
    print(f"HTML file saved")